# FMCSA SMS Data Cleaning Notebook

This notebook cleans and filters public FMCSA SMS input data for the Upper Midwest Fleet Safety & DOT Compliance Risk Dashboard.

The workflow creates cleaned carrier, crash, inspection, and violation tables for carriers based in ND, MN, SD, IA, and WI.

Large raw FMCSA source files are not included in this repository due to file size.

## Data Files

This notebook expects the raw FMCSA SMS input CSV files to be stored in a local `data_raw` folder.

Raw source files are not included in this repository due to file size. Cleaned output files are written to a local `data_clean` folder.

In [ ]:
# =====================================================
# 1. Project Setup
# =====================================================

from pathlib import Path
import pandas as pd

# Project folders
project_folder = Path(".")
data_raw = project_folder / "data_raw"
data_clean = project_folder / "data_clean"

# Crash file
crash_file = data_raw / "SMS_Input_-_Crash_20260529.csv"

# Check paths
print("Project folder exists:", project_folder.exists())
print("Raw folder exists:", data_raw.exists())
print("Clean folder exists:", data_clean.exists())
print("Crash file exists:", crash_file.exists())

In [ ]:
size_mb = crash_file.stat().st_size / (1024 * 1024)
print(f"Crash file size: {size_mb:,.2f} MB")

In [ ]:
crash_preview = pd.read_csv(crash_file, nrows=5, low_memory=False)

print("Column names:")
print(crash_preview.columns.tolist())

crash_preview.head()

In [ ]:
row_count = sum(1 for _ in open(crash_file, encoding="utf-8", errors="ignore")) - 1
print(f"Total crash rows: {row_count:,}")

In [ ]:
# ============================================================================
# 2. Clean Crash Data
# ============================================================================

import pandas as pd
from pathlib import Path

# Project folders
project_folder = Path(r"C:\Users\nkc06\OneDrive\Desktop\Fleet_Safety_Dashboard")
data_raw = project_folder / "data_raw"
data_clean = project_folder / "data_clean"

# Files
crash_file = data_raw / "SMS_Input_-_Crash_20260529.csv"
clean_crash_file = data_clean / "crashes_midwest.csv"

# States for the dashboard
midwest_states = ["ND", "MN", "SD", "IA", "WI"]

# Read crash data
crashes = pd.read_csv(crash_file, low_memory=False)

# Check state counts before filtering
print("Crash counts by state before filtering:")
print(crashes["Report_State"].value_counts().head(20))

# Filter to Upper Midwest states
crashes_midwest = crashes[crashes["Report_State"].isin(midwest_states)].copy()

# Save cleaned crash file
crashes_midwest.to_csv(clean_crash_file, index=False)

print("\nRows before filtering:", len(crashes))
print("Rows after filtering:", len(crashes_midwest))
print("Saved file to:", clean_crash_file)

In [ ]:
crashes_midwest.head()

In [ ]:
crashes_midwest["Report_State"].value_counts()

In [ ]:
# Convert Report_Date to a real date field
crashes_midwest["Report_Date"] = pd.to_datetime(
    crashes_midwest["Report_Date"],
    format="%d-%b-%y",
    errors="coerce"
)

# Add date helper columns for Power BI
crashes_midwest["Crash_Year"] = crashes_midwest["Report_Date"].dt.year
crashes_midwest["Crash_Month"] = crashes_midwest["Report_Date"].dt.month
crashes_midwest["Crash_Month_Name"] = crashes_midwest["Report_Date"].dt.strftime("%b")
crashes_midwest["Crash_Quarter"] = "Q" + crashes_midwest["Report_Date"].dt.quarter.astype(str)

# Save updated clean crash file
crashes_midwest.to_csv(clean_crash_file, index=False)

print("Updated crash file saved.")
print(crashes_midwest[["Report_Date", "Crash_Year", "Crash_Month", "Crash_Month_Name", "Crash_Quarter"]].head())

In [ ]:
# ============================================================
# 3. Clean Census / Carrier Data
# ============================================================

from pathlib import Path
import pandas as pd

# Project folders
project_folder = Path(r"C:\Users\nkc06\OneDrive\Desktop\Fleet_Safety_Dashboard")
data_raw = project_folder / "data_raw"
data_clean = project_folder / "data_clean"

# Census file
census_file = data_raw / "SMS_Input_-_Motor_Carrier_Census_Information_20260529.csv"

# Check file
print("Census file exists:", census_file.exists())

size_mb = census_file.stat().st_size / (1024 * 1024)
print(f"Census file size: {size_mb:,.2f} MB")

In [ ]:
census_preview = pd.read_csv(census_file, nrows=5, low_memory=False)

print("Column names:")
print(census_preview.columns.tolist())

census_preview.head()

In [ ]:
# Read only the state column first
state_counts = pd.read_csv(
    census_file,
    usecols=["PHY_STATE"],
    dtype=str,
    low_memory=False
)["PHY_STATE"].value_counts()

print(state_counts.head(20))

In [ ]:
# ============================================================
# Filter Census Data to Upper Midwest Carriers
# ============================================================

midwest_states = ["ND", "MN", "SD", "IA", "WI"]

# Columns we want for the carrier dimension table
carrier_columns = [
    "DOT_NUMBER",
    "LEGAL_NAME",
    "DBA_NAME",
    "CARRIER_OPERATION",
    "HM_FLAG",
    "PC_FLAG",
    "PHY_CITY",
    "PHY_STATE",
    "PHY_ZIP",
    "PHY_COUNTRY",
    "MAILING_CITY",
    "MAILING_STATE",
    "MCS150_DATE",
    "MCS150_MILEAGE",
    "MCS150_MILEAGE_YEAR",
    "ADD_DATE",
    "OIC_STATE",
    "NBR_POWER_UNIT",
    "DRIVER_TOTAL",
    "RECENT_MILEAGE",
    "RECENT_MILEAGE_YEAR"
]

# Read only the useful columns
carriers = pd.read_csv(
    census_file,
    usecols=carrier_columns,
    dtype=str,
    low_memory=False
)

# Filter to Midwest states
carriers_midwest = carriers[carriers["PHY_STATE"].isin(midwest_states)].copy()

# Remove duplicate DOT numbers so this can become DimCarrier
carriers_midwest = carriers_midwest.drop_duplicates(subset=["DOT_NUMBER"])

# Save clean carrier dimension file
clean_carrier_file = data_clean / "dim_carrier_midwest.csv"
carriers_midwest.to_csv(clean_carrier_file, index=False)

print("Rows before filtering:", len(carriers))
print("Rows after filtering:", len(carriers_midwest))
print("Saved file to:", clean_carrier_file)

print("\nCarrier counts by state:")
print(carriers_midwest["PHY_STATE"].value_counts())

In [ ]:
carriers_midwest.head()

In [ ]:
carriers_midwest["DOT_NUMBER"].duplicated().sum()

In [ ]:
# ============================================================
# 4. Clean Inspection Data
# ============================================================

inspection_file = data_raw / "SMS_Input_-_Inspection_20260529.csv"

print("Inspection file exists:", inspection_file.exists())

size_mb = inspection_file.stat().st_size / (1024 * 1024)
print(f"Inspection file size: {size_mb:,.2f} MB")

In [ ]:
inspection_preview = pd.read_csv(inspection_file, nrows=5, low_memory=False)

print("Column names:")
print(inspection_preview.columns.tolist())

inspection_preview.head()

In [ ]:
# ============================================================
# Filter Inspection Data to Midwest Carriers using chunks
# ============================================================

import pandas as pd

# Create a set of Midwest DOT numbers from DimCarrier
midwest_dot_numbers = set(carriers_midwest["DOT_NUMBER"].astype(str))

# Get actual column names from the file
actual_inspection_columns = pd.read_csv(
    inspection_file,
    nrows=0,
    low_memory=False
).columns.tolist()

# Columns we would like to keep
requested_inspection_columns = [
    "Unique_ID",
    "Report_Number",
    "Report_State",
    "DOT_Number",
    "Insp_Date",
    "Insp_Level_ID",
    "County_code_State",
    "Time_Weight",
    "Driver_OOS_Total",
    "Vehicle_OOS_Total",
    "Total_Hazmat_Sent",
    "OOS_Total",
    "Hazmat_OOS_Total",
    "Hazmat_Placard_req",
    "Unit_Type_Desc",
    "Unit_Make",
    "Unsafe_Insp",
    "Fatigued_Insp",
    "Dr_Fitness_Insp",
    "Subt_Alcohol_Insp",
    "Vh_Maint_Insp",
    "HM_Insp",
    "BASIC_Viol",
    "Unsafe_Viol",
    "Fatigued_Viol",
    "Dr_Fitness_Viol",
    "Subt_Alcohol_Viol",
    "Vh_Maint_Viol",
    "HM_Viol"
]

# Keep only columns that actually exist
inspection_columns = [
    col for col in requested_inspection_columns
    if col in actual_inspection_columns
]

missing_columns = [
    col for col in requested_inspection_columns
    if col not in actual_inspection_columns
]

print("Columns we will use:")
print(inspection_columns)

print("\nMissing columns:")
print(missing_columns)

# Output file
clean_inspection_file = data_clean / "fact_inspections_midwest.csv"

# Chunk settings
chunk_size = 250_000
first_chunk = True
total_rows_read = 0
total_rows_kept = 0

for chunk in pd.read_csv(
    inspection_file,
    usecols=inspection_columns,
    dtype=str,
    chunksize=chunk_size,
    low_memory=False
):
    total_rows_read += len(chunk)

    # Filter to Midwest-based carriers
    chunk_midwest = chunk[chunk["DOT_Number"].isin(midwest_dot_numbers)].copy()
    total_rows_kept += len(chunk_midwest)

    # Append filtered rows to clean CSV
    chunk_midwest.to_csv(
        clean_inspection_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    print(f"Rows read: {total_rows_read:,} | Rows kept: {total_rows_kept:,}")

print("\nDone.")
print("Total rows read:", f"{total_rows_read:,}")
print("Total rows kept:", f"{total_rows_kept:,}")
print("Saved file to:", clean_inspection_file)

In [ ]:
inspections_midwest = pd.read_csv(clean_inspection_file, dtype=str, low_memory=False)

print("Clean inspection rows:", len(inspections_midwest))
print("Duplicate Unique_ID count:", inspections_midwest["Unique_ID"].duplicated().sum())

inspections_midwest.head()

In [ ]:
# Convert inspection date to real date
inspections_midwest["Insp_Date"] = pd.to_datetime(
    inspections_midwest["Insp_Date"],
    format="%d-%b-%y",
    errors="coerce"
)

# Add helper date columns for Power BI
inspections_midwest["Inspection_Year"] = inspections_midwest["Insp_Date"].dt.year
inspections_midwest["Inspection_Month"] = inspections_midwest["Insp_Date"].dt.month
inspections_midwest["Inspection_Month_Name"] = inspections_midwest["Insp_Date"].dt.strftime("%b")
inspections_midwest["Inspection_Quarter"] = "Q" + inspections_midwest["Insp_Date"].dt.quarter.astype(str)

# Save updated inspection file
inspections_midwest.to_csv(clean_inspection_file, index=False)

print("Updated inspection file saved.")
print(inspections_midwest[[
    "Unique_ID",
    "DOT_Number",
    "Insp_Date",
    "Inspection_Year",
    "Inspection_Month",
    "Inspection_Month_Name",
    "Inspection_Quarter"
]].head())

In [ ]:
# ============================================================
# 5. Clean Violation Data
# ============================================================

violation_file = data_raw / "SMS_Input_-_Violation_20260529.csv"

print("Violation file exists:", violation_file.exists())

size_mb = violation_file.stat().st_size / (1024 * 1024)
print(f"Violation file size: {size_mb:,.2f} MB")

In [ ]:
violation_preview = pd.read_csv(violation_file, nrows=5, low_memory=False)

print("Column names:")
print(violation_preview.columns.tolist())

violation_preview.head()

In [ ]:
inspections_midwest["Unique_ID"]

In [ ]:
# ============================================================
# Filter Violation Data to Midwest Inspections using chunks
# ============================================================

# Create a set of Midwest inspection IDs
midwest_inspection_ids = set(inspections_midwest["Unique_ID"].astype(str))

# Columns to keep for the violation fact table
violation_columns = [
    "Unique_ID",
    "Insp_Date",
    "DOT_Number",
    "Viol_Code",
    "BASIC_Desc",
    "OOS_Indicator",
    "OOS_Weight",
    "Severity_Weight",
    "Time_Weight",
    "Total_Severity_Wght",
    "Section_Desc",
    "Group_Desc",
    "Viol_Unit"
]

# Output file
clean_violation_file = data_clean / "fact_violations_midwest.csv"

# Chunk settings
chunk_size = 250_000
first_chunk = True
total_rows_read = 0
total_rows_kept = 0

for chunk in pd.read_csv(
    violation_file,
    usecols=violation_columns,
    dtype=str,
    chunksize=chunk_size,
    low_memory=False
):
    total_rows_read += len(chunk)

    # Filter to violations tied to Midwest inspections
    chunk_midwest = chunk[
        chunk["Unique_ID"].isin(midwest_inspection_ids)
    ].copy()

    total_rows_kept += len(chunk_midwest)

    # Append filtered rows to clean CSV
    chunk_midwest.to_csv(
        clean_violation_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    print(f"Rows read: {total_rows_read:,} | Rows kept: {total_rows_kept:,}")

print("\nDone.")
print("Total rows read:", f"{total_rows_read:,}")
print("Total rows kept:", f"{total_rows_kept:,}")
print("Saved file to:", clean_violation_file)

In [ ]:
violations_midwest = pd.read_csv(clean_violation_file, dtype=str, low_memory=False)

print("Clean violation rows:", len(violations_midwest))
print("Unique inspections with violations:", violations_midwest["Unique_ID"].nunique())
print("Unique DOT numbers:", violations_midwest["DOT_Number"].nunique())

violations_midwest.head()

In [ ]:
print("Violation counts by BASIC category:")
print(violations_midwest["BASIC_Desc"].value_counts())

In [ ]:
# Rename/save crash file using fact table naming pattern
clean_crash_file_final = data_clean / "fact_crashes_midwest.csv"

crashes_midwest.to_csv(clean_crash_file_final, index=False)

print("Saved final crash fact file to:", clean_crash_file_final)

In [ ]:
for file in data_clean.glob("*.csv"):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file.name}: {size_mb:,.2f} MB")

In [ ]:
inspection_ids = set(inspections_midwest["Unique_ID"].astype(str))
violation_ids = set(violations_midwest["Unique_ID"].astype(str))

missing_from_inspections = violation_ids - inspection_ids

print("Violation inspection IDs missing from inspections:", len(missing_from_inspections))

In [ ]:
carrier_dot_numbers = set(carriers_midwest["DOT_NUMBER"].astype(str))
inspection_dot_numbers = set(inspections_midwest["DOT_Number"].astype(str))

missing_from_carriers = inspection_dot_numbers - carrier_dot_numbers

print("Inspection DOT numbers missing from carriers:", len(missing_from_carriers))